In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task1 read data:
# Load the dataset
delivery_path = os.path.join(path,'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)

print(f"Dataset shape: {df_delivery.shape}")

In [ ]:
# Task 2: using head to show first rows data:
df_delivery.head()

In [ ]:
# Task 3: display info to show information of data:
df_delivery.info()

In [ ]:
# Task 4: using describe to know the details of data
df_delivery.describe()

In [ ]:
# Task 5:
# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: drop ID column
print(f"Before: {df_delivery.shape}")
df_delivery = df_delivery.drop(columns=['Order_ID'])
print(f"After dropping Order_ID column: {df_delivery.shape}")

df_delivery

In [ ]:
# Task 2: check the missing
def check_missing_values(df_delivery):
  missing_values = df_delivery.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery)

at_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']
# Drop rows with missing stat values
df_delivery = df_delivery.dropna(subset=at_cols).copy()
print(f"Shape after cleaning: {df_delivery.shape}")


In [ ]:
# Task 3: Dublicate
def check_duplicates(df_delivery):
  duplicates = df_delivery.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_delivery.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)

In [ ]:
# Task 4: Do we have categorical columns?
# encoding
categorical_cols = df_delivery.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df_delivery[col] = le.fit_transform(df_delivery[col])
df_delivery


In [ ]:
# Task 5:
numerical_cols = df_delivery.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_delivery[numerical_cols] = scaler.fit_transform(df_delivery[numerical_cols])
df_delivery.head()


In [ ]:
# Task 6:
def check_target_imbalance(df_delivery, target_column):
  print("Target Distribution:")
  print(df_delivery[target_column].value_counts(normalize=True))
  sns.countplot(x=df_delivery[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_delivery, "Delivery_Time")

In [ ]:
# Task 1:
feature_cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
X = df_delivery[feature_cols]
y = df_delivery['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

In [ ]:
# Task 2,3,4,5: Write your code here:
#task3
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

#MSE
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  ${mae:,.2f}")

#
# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)






In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.xlabel("Actual Dilvery time (Ground Truth)")
plt.ylabel("Predicted Dilvery time")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: